<table>
  <tr>
    <td><div align="left"><font size="30">Planar homography</font></div></td>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/docs/figs/VisionToolboxLogo_NoBackgnd@2x.png?raw=1" width="200"></td>
  </tr>
</table>

(c) Peter Corke 2024

Robotics, Vision & Control: Python, see Section 13.6.2

## Configuring the Jupyter environment
We need to import some packages to help us with linear algebra (`numpy`), graphics (`matplotlib`), and machine vision (`machinevisiontoolbox`).
If you're running locally you need to have these packages installed.  If you're running on CoLab we have to first install machinevisiontoolbox which is not preinstalled, this will be a bit slow.

In [ ]:
# MVTB_BOOTSTRAP_CELL -- sets up the environment (Colab / JupyterLite / local install); click to expand. Generated from docs/notebooks/_mvtb_nb_bootstrap.py by sync_bootstrap.py -- do not hand-edit.

"""Environment bootstrap for machinevision-toolbox-python's Jupyter notebooks.

Installs the toolbox (and reports how) across the three environments a notebook in
this folder might run in: a local Jupyter/VS Code install, Google Colab, and
JupyterLite (Pyodide/WASM, in-browser).

This file is the single source of truth for that logic. Every notebook's own
bootstrap cell is a generated copy of this file's content, produced by
sync_bootstrap.py -- see docs/notebooks/README.md for the full explanation.
"""

import subprocess
import sys
from pathlib import Path


async def ensure_installed() -> bool:
    """Install machinevision-toolbox-python if needed, and report the environment.

    :returns: True if running on Google Colab, False otherwise.
    """
    if sys.platform == "emscripten":
        import micropip

        await micropip.install(
            [
                "opencv-python",
                "spatialmath-python",
                "pgraph-python",
                "ansitable",
                "mvtb-data",
                "tqdm",
                "requests",
            ]
        )
        import cv2  # noqa: F401 - force cv2 into module registry before toolbox import

        wheels = sorted(Path("/pypi").glob("machinevision_toolbox_python-*.whl"))
        if wheels:
            # Prefer the wheel bundled with this JupyterLite site.
            await micropip.install(wheels[-1].as_posix(), deps=False)
        else:
            # Fall back to PyPI when running outside the published site layout.
            await micropip.install("machinevision-toolbox-python", deps=False)
        where, colab = "in browser", False
    else:
        try:
            import google.colab  # noqa: F401
        except ImportError:
            where, colab = "locally", False
        else:
            print("Installing machinevision-toolbox-python...")
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "machinevision-toolbox-python",
                ],
                check=True,
            )
            where, colab = "on Colab", True

    import machinevisiontoolbox

    version = getattr(machinevisiontoolbox, "__version__", "unknown")
    print(f"Running {where} using MVTB v{version}")
    return colab

COLAB = await ensure_installed()


In [ ]:
%matplotlib ipympl

import numpy as np
np.set_printoptions(linewidth=120, formatter={'float': lambda x: f"{x:8.4g}" if abs(x) > 1e-10 else f"{0:8.4g}"})

from spatialmath import SE3
from spatialmath.base import e2h, h2e, homtrans, plot_sphere
from machinevisiontoolbox import CentralCamera


***

We define a central perspective camera (see camera.ipynb for more details on this), that is position up high, looking obliquely downward at the ground

In [ ]:
camera = CentralCamera(f=0.012, rho=10e-6, imagesize=1000, 
        pose=SE3(0, 0, 8) * SE3.Rx(-2.8))

And we can plot the camera in the 3D world

In [ ]:
ax = camera.plot(scale=2, shape='camera', color='k', frame=True)
ax.set_xlim(-8, 12)
ax.set_ylim(-10, 10)
ax.set_zlim(0, 10)

A shape on the ground plane is defined by a set of 2D coordinates

In [ ]:
P = np.column_stack([[-1, 1], [-1, 2], [ 2,2], [2, 1]])
P

To obtain the coordinates of the points in 3D, we augment each column with a zero, since the ground plane is defined by $z=0$

In [ ]:
P0 = np.vstack([P, np.zeros((4,))])
P0

Now we can project the 3D ground plane points onto the image plane

In [ ]:
camera.project_point(P0)

The homography is computed from the camera matrix by deleting column two (the z column)

In [ ]:
H = np.delete(camera.C(), 2, axis=1)
H

We can use this matrix to directly compute the image plane points, by transforming the homogeneous ground plane points

In [ ]:
h2e(H @ e2h(P))

or more simply

In [ ]:
homtrans(H, P)

which first converts `P` to homogeneous form, performs the multiplication, then converts the resulting homogeneous coordinates to Euclidean.


H is square and of full rank, so it is invertible. This means that we can perform the inverse mapping, from the image plane
to the ground plane.

The camera has a 1000 x 1000 image plane so the coordinates of its corners are


In [ ]:
p = np.column_stack([[0, 0], [0, 1000], [1000, 1000], [1000, 0]])
p

and on the ground plane these are the points

In [ ]:
Pi = homtrans(np.linalg.inv(H), p)
Pi

Now we can overlay the corners of the camera's field of view onto the "world view" of the imaging setup that we showed earlier

In [ ]:
ax = camera.plot(scale=2, shape='camera', color='k', frame=True)
k = [0, 1, 2, 3, 0]
ax.plot(Pi[0, k], Pi[1, k], np.zeros(5), 'b--')


Scroll back up to the previous figure to see the blue dashed line representing the field of view.